<img src='./img/cng_thrivegeo_banner.png' class="center" width=95%>

# 🚀 Discover (Geo)Parquet

### Introduction

So far, you have worked with raster data: images and arrays of pixels. Much geospatial data is **vector** data instead: points, lines and polygons such as buildings, roads or field boundaries, each with a table of attributes. These datasets can be huge. Overture Maps alone holds more than two billion building footprints.

Traditional formats such as Shapefile, GeoJSON or CSV store data **row by row**, one feature after another. To read a single attribute, or only the features in one area, you often have to read the whole file.

[Apache Parquet](https://parquet.apache.org/) is a file format for large tables, widely used in data engineering. Three features make it cloud-friendly:

* 📋 **Footer metadata**: the schema and statistics sit at the end of the file, so one small request tells you what's inside
* 🧱 **Row groups**: rows are stored in blocks, and each block records statistics (minimum, maximum, number of empty values), so a reader can skip blocks that don't match a filter
* 📊 **Columns**: within each block, data is stored column by column, so you can read only the attributes you need

[GeoParquet](https://geoparquet.org/) adds a standard way to store geometries and their coordinate reference system (CRS) in Parquet, so GIS tools recognise the file as spatial data.

The pattern is the one you know from COG and Zarr: **read the description first, then fetch only the pieces you need.** Throughout, you keep an eye on one number: **how much of a file you actually read**.

Here, you work with two datasets:

* 🗺️ **Overture Maps**: open vector datasets of the world, stored as GeoParquet on S3. You find its latest release, browse its folders, read the footer of a file, and see which part of the world each block of a file covers.
* 🌾 **EuroCrops**: the field boundaries of European farms. You apply what you learnt to map vineyards in Austria.

**Prerequisites:** basic Python and *Discover STAC*. *Discover COG* and *Discover Zarr* help, but are not required.

### By the end of this tutorial, you will be able to...

* 💡 **Explain** how the footer, row groups and columns let you read only part of a Parquet file
* 🗂️ **Browse** a partitioned dataset on S3 and read folder names such as `theme=base/type=land_use`
* 📋 **Read** a Parquet file's footer, schema and `geo` metadata with `pyarrow`, without loading any rows
* ✂️ **Use** row-group statistics to see which area each block covers, and read only the row groups and columns you need
* 🌍 **Turn** GeoParquet data into a GeoDataFrame, map it and save it

### Notebook Outline

* [Example 1️⃣: Look before you load](#example1)
* [Example 2️⃣: Load the buildings of Snowbird](#example2)
* [Exercise 1️⃣: Vineyards from EuroCrops](#example3)
* [Takeaways](#example4)

<hr>

**Import required libraries**

In [1]:
import json

import geopandas as gpd
import pandas as pd
import pyarrow.parquet as pq
import s3fs
from pystac_client import Client
from shapely.geometry import box

::: {.callout-note}
## One filesystem for everything

`s3fs` lets you browse an S3 bucket as if it were a folder on your computer. You will use the same `s3fs` connection both to list the folders and to open the Parquet files with `pyarrow`.
:::

<hr>

<a id="example1"></a>

## Example 1️⃣: Look before you load

[Overture Maps](https://overturemaps.org/) provides comprehensive open datasets across multiple **themes**:
<center>

| Theme | What it holds |
|---|---|
| `addresses` | address points |
| `base` | land cover, land use, water and infrastructure |
| `buildings` | more than 2 billion building footprints |
| `divisions` | administrative boundaries, from countries to neighbourhoods |
| `places` | points of interest, such as shops and landmarks |
| `transportation` | roads, paths and railways |

</center>


All of it is stored as GeoParquet on **S3**. As with COGs and Zarr, you can find out a great deal before you read any of it. In this example, you:

1. 🔌 **Find** the latest release through STAC
2. 🗂️ **Browse** the folders
3. 📋 **Read** the footer of one file
4. 🧬 **Look** at its columns and its `geo` entry
5. 👀 **Read** your first rows

### 🔌 Step 1: Find the latest release

Overture publishes a new release every month. Each release is a complete, new copy of the data, stored in its own folder. To make sure your code always points at the current data, you can ask Overture's STAC catalog which release is the latest:

In [2]:
om_stac = Client.open("https://stac.overturemaps.org/catalog.json")
om_stac

/opt/conda/lib/python3.13/site-packages/pystac_client/client.py:191: NoConformsTo: Server does not advertise any conformance classes.
  warnings.warn(NoConformsTo())


<Client id=Overture Releases>

Besides the standard STAC fields, providers can add their own information to a catalog. `pystac` collects these extra fields in `extra_fields`. Overture uses one of them, `latest`, to name the most recent release:

In [3]:
latest_release = om_stac.extra_fields["latest"]
release_path = f"overturemaps-us-west-2/release/{latest_release}"

print("Latest release:", latest_release)
print("Stored in:     ", release_path)

Latest release: 2026-09-23.1
Stored in:      overturemaps-us-west-2/release/2026-09-23.1


### 🗂️ Step 2: Browse the folders

The release folder lives in a public S3 bucket. With `s3fs`, you can list its contents just like a folder on your computer. `anon=True` connects anonymously, because the bucket is public.

`s3.ls()` returns full paths, so you keep only the last part of each path, the folder name:

In [4]:
s3 = s3fs.S3FileSystem(anon=True, client_kwargs={"region_name": "us-west-2"})

[path.split("/")[-1] for path in s3.ls(release_path)]

['theme=addresses',
 'theme=base',
 'theme=buildings',
 'theme=divisions',
 'theme=places',
 'theme=transportation']

There is one folder per theme, and each folder name has the form **`key=value`**: in `theme=buildings`, the key is `theme` and the value is `buildings`.

This naming style is called **hive-style partitioning**. The folder names describe what is inside, so many tools, such as DuckDB, can read them as if they were columns and skip whole folders that don't match a filter.

Go one level deeper, into the `base` theme:

In [5]:
base_path = f"{release_path}/theme=base"

[path.split("/")[-1] for path in s3.ls(base_path)]

['type=bathymetry',
 'type=infrastructure',
 'type=land',
 'type=land_cover',
 'type=land_use',
 'type=water']

The same pattern repeats: each theme is split into **types**, again with `key=value` folders. So the full layout of an Overture release is:

<center>

```text
s3://overturemaps-us-west-2/release/<RELEASE>/theme=<THEME>/type=<TYPE>/*.parquet
```
</center>
Now go into one type, the land-use polygons (parks, farmland, residential areas and so on), and look at the files:

In [6]:
land_use_path = f"{base_path}/type=land_use"
land_use_files = s3.ls(land_use_path)
land_use_files
print(f"{len(land_use_files)} Parquet files")
print(f"{s3.du(land_use_path) / 1e9:.0f} GB in total")
print("First file:", land_use_files[0].split("/")[-1])

64 Parquet files
19 GB in total
First file: part-00000-18dcb239-f29e-5e57-b36f-4bd46c12fb86-c000.zstd.parquet


::: {.callout-tip}

## Reflection and answers

Imagine you want the land-use polygons of norther Portugal, where you worked in *Discover Zarr*.

- ➡️ 1. Which folders do you need to open? Which can you skip completely?<br>
    ✏️ : You only need `theme=base/type=land_use`. All the other themes and types are skipped without opening a single file. This is the first level of skipping: **whole folders**.
- ➡️ 2. Can you tell from the file names which of the land-use files contain Portugal?<br>
    ✏️ : No. The file names are just numbers. To find out which files, and which parts of a file, cover Portugal, you need to look **inside** the files. That is what you will do in the next two examples.

:::


The provider decides how a large dataset is split into files and folders, and chooses a layout that matches the most common questions. Overture splits by **theme** and **type**, because most users want one kind of feature, e.g. only buildings or only roads.

You met the same idea in the Zarr notebook: *chunks follow the question*. Here, **partitions follow the question**.

### 📋 Step 3: Read the footer

The file names don't tell you what is inside. A Parquet file keeps its description in a **footer** at the end of the file. Pick the first land-use file. `pq.read_metadata()` reads only the footer:

In [7]:
land_use_file = land_use_files[0]

footer = pq.read_metadata(land_use_file, filesystem=s3)
footer

  created_by: parquet-cpp-arrow version 17.0.0
  num_columns: 35
  num_rows: 884912
  num_row_groups: 64
  format_version: 2.6
  serialized_size: 284725

Here is what each line tells you:
<center>

| Line | Meaning |
|---|---|
| **num_columns** | the number of columns stored in the file |
| **num_rows** | the number of rows, i.e. features |
| **num_row_groups** | the number of blocks the rows are split into |
| **created_by** | the software that wrote the file |
| **serialized_size** | the size of the footer itself, in bytes |

</center>
How much of the file did you just read?

In [8]:
file_size = s3.size(land_use_file)

print(f"File size:   {file_size / 1e6:.0f} MB")
print(f"Footer size: {footer.serialized_size / 1e3:.0f} KB")
print(f"Share read:  {footer.serialized_size / file_size:.3%}")

File size:   319 MB
Footer size: 285 KB
Share read:  0.089%


A tiny share of the file told you how many features it holds and how they are organised, just like the header of a COG or the metadata files of a Zarr store.

### 🧬 Step 4: Look at the columns and the `geo` entry

To work with the file further, open it with `pq.ParquetFile()`. This also reads only the footer. Its `schema_arrow` lists the columns and their data types:

In [9]:
land_use = pq.ParquetFile(land_use_file, filesystem=s3)
land_use.schema_arrow

id: string
geometry: binary
sources: list<element: struct<property: string, dataset: string, license: string, record_id: string, update_t (... 118 chars omitted)
  child 0, element: struct<property: string, dataset: string, license: string, record_id: string, update_time: string, c (... 103 chars omitted)
      child 0, property: string
      child 1, dataset: string
      child 2, license: string
      child 3, record_id: string
      child 4, update_time: string
      child 5, confidence: double
      child 6, between: list<element: double>
          child 0, element: double
      child 7, provider: string
      child 8, resource: string
      child 9, version: string
names: struct<primary: string, common: map<string, string ('common')>, rules: list<element: struct<variant: (... 159 chars omitted)
  child 0, primary: string
  child 1, common: map<string, string ('common')>
      child 0, common: struct<key: string not null, value: string> not null
          child 0, key: string not n

Most columns hold simple values, such as text (`string`) or numbers (`double`). Two columns are worth a closer look:

* `bbox` is a `struct`: a group of values stored together. It holds the **bounding box** of each feature, as `xmin`, `xmax`, `ymin` and `ymax`. You will use it in the next example.
* `geometry` holds the shape of each feature.

::: {.callout-tip}
## Reflection and answers

- ➡️ 1. Find the `geometry` column in the schema. What data type does it have?<br>
    ✏️ : `binary`: a sequence of raw bytes.
- ➡️ 2. Why do you think the geometry is stored this way?<br>
    ✏️ : Parquet was designed for ordinary tables. It has no data type for points, lines or polygons. So each geometry is stored in a standard binary format called **WKB** (well-known binary), which any GIS tool can decode.
:::

A **GeoParquet** file is a normal Parquet file with one extra piece of information in its footer: a `geo` entry. You can already see it at the very end of the schema above. It tells GIS tools which column holds the geometry, how it is encoded and which coordinate reference system (CRS) it uses:

In [10]:
geo = json.loads(land_use.schema_arrow.metadata[b"geo"])
geometry_info = geo["columns"][geo["primary_column"]]

crs = geometry_info.get("crs")
crs_name = crs["name"] if isinstance(crs, dict) else "not given, so longitude and latitude in degrees"

print("GeoParquet version:", geo.get("version"))
print("Geometry column:   ", geo["primary_column"])
print("Encoding:          ", geometry_info["encoding"])
print("Geometry types:    ", geometry_info.get("geometry_types"))
print("CRS:               ", crs_name)
print("Bounding box in:   ", list(geometry_info.get("covering", {})))

GeoParquet version: 1.1.0
Geometry column:    geometry
Encoding:           WKB
Geometry types:     ['LineString', 'MultiPolygon', 'Polygon']
CRS:                not given, so longitude and latitude in degrees
Bounding box in:    ['bbox']


<center>

| Line | Meaning |
|---|---|
| **Geometry column** | the column that holds the geometries |
| **Encoding** | how the geometries are stored: `WKB`, as you saw above |
| **Geometry types** | which kinds of geometry the file holds, e.g. polygons |
| **CRS** | the coordinate reference system. If none is given, GeoParquet means longitude and latitude in degrees |
| **Bounding box in** | the column that holds each feature's bounding box: the `bbox` column |

</center>

Without the `geo` entry, a GIS tool would only see a column of bytes. With it, the file becomes spatial data.

### 🗺️ Step 5: Read the land use of Salt Lake City

So far, you have read only footers. Now read some data, but only what you need: the land-use polygons of central Salt Lake City. `gpd.read_parquet()` accepts a `bbox`. This works because of the `geo` entry you just read: it names the `bbox` column, so the reader knows where to look.

In [11]:
%%time
slc_bbox = (-111.95, 40.70, -111.82, 40.80)   # west, south, east, north

slc_land_use = gpd.read_parquet(
    land_use_path,
    filesystem=s3,
    columns=["subtype", "class", "geometry"],
    bbox=slc_bbox,
)

print(len(slc_land_use), "land-use polygons")

5660 land-use polygons
CPU times: user 1.61 s, sys: 470 ms, total: 2.08 s
Wall time: 1min 34s


In [ ]:
slc_land_use.explore(column="class", tooltip=["subtype", "class"])

<hr>

<a id="example2"></a>

## Example 2️⃣: Load the buildings of Snowbird

In Example 1, the reader found Salt Lake City for you. Now you do the same search by hand, to see how it works. Your question: **which buildings stand in and around Snowbird**, the ski resort from *Discover STAC* and *Discover COG*?

Overture Maps holds more than two billion buildings. You narrow them down level by level, and at each level you note how much is left:

1. 🗂️ **Folder**: go to the buildings
2. 🔎 **File**: find the file that covers Snowbird
3. 🧱 **Row groups**: find the blocks that cover Snowbird
4. 🗺️ **Read** and map the result

### 🗂️ Step 1: Go to the buildings folder

The folder names take you straight to the buildings. Everything else in the release is skipped:

In [13]:
snowbird_bbox = (-111.724091, 40.517975, -111.493378, 40.640530)   # west, south, east, north
snowbird = box(*snowbird_bbox)

building_path = f"{release_path}/theme=buildings/type=building"
building_files = s3.ls(building_path)
total_size = s3.du(building_path)

print(f"{len(building_files)} files, {total_size / 1e9:.0f} GB")

512 files, 277 GB


### 🔎 Step 2: Find the file that covers Snowbird

The file names are just numbers. But Overture's STAC catalog includes an **index**: one small Parquet file that lists every data file of the release with its bounding box. Read it, and keep the rows for buildings:

In [14]:
index = pd.read_parquet(f"https://stac.overturemaps.org/{latest_release}/collections.parquet")
building_index = index[index["collection"] == "building"]

footprints = gpd.GeoDataFrame(
    {"file": [assets["aws"]["alternate"]["s3"]["href"] for assets in building_index["assets"]]},
    geometry=[box(b["xmin"], b["ymin"], b["xmax"], b["ymax"]) for b in building_index["bbox"]],
    crs="EPSG:4326",
)

covering_files = footprints[footprints.intersects(snowbird)]
print(len(covering_files), "of", len(footprints), "files cover Snowbird")

1 of 512 files cover Snowbird


Each row of `footprints` is one file and the area it covers. Draw them all, with the files that cover Snowbird in red:

In [ ]:
m = footprints.explore(style_kwds={"fillOpacity": 0.05})
covering_files.explore(m=m, color="red")

::: {.callout-tip}
## Reflection and answers

- ➡️ 1. Does each file cover a compact area, or is it spread over the whole world? What does this tell you about how Overture wrote the files?<br>
    ✏️ : Each file covers a **compact area**. Overture **sorts** the buildings by location before writing them, so buildings that are close together end up in the same file. Without sorting, every file would hold buildings from all over the world, and none could be skipped.
- ➡️ 2. How many of the files do you still need?<br>
    ✏️ : Only the one in red. All the others are skipped without being opened.

:::

### 🧱 Step 3: Find the row groups that cover Snowbird

Open the file. As in Example 1, this reads only its footer:

In [16]:
building_file = covering_files["file"].iloc[0].removeprefix("s3://")
buildings = pq.ParquetFile(building_file, filesystem=s3)
file_size = s3.size(building_file)

print(f"File size:  {file_size / 1e6:.0f} MB")
print("Rows:      ", buildings.metadata.num_rows)
print("Row groups:", buildings.metadata.num_row_groups)

File size:  666 MB
Rows:       4865057
Row groups: 256


The rows of the file are split into **row groups**. For **every column in every row group**, the footer stores a few **statistics**: the smallest value, the largest value and the number of empty values. Look at the statistics of the `bbox` column for the first row group:

In [17]:
first_group = buildings.metadata.row_group(0)
print(first_group)

for i in range(first_group.num_columns):
    column = first_group.column(i)
    if column.path_in_schema.startswith("bbox."):
        print(f"{column.path_in_schema}:  min {column.statistics.min:9.3f}   max {column.statistics.max:9.3f}")

  num_columns: 45
  num_rows: 30894
  total_byte_size: 6710665
  sorting_columns: ()
bbox.xmin:  min  -117.405   max  -115.271
bbox.xmax:  min  -117.405   max  -115.270
bbox.ymin:  min    35.077   max    36.053
bbox.ymax:  min    35.077   max    36.053


Inside the file, `bbox` is stored as four separate columns. The smallest `bbox.xmin` and `bbox.ymin` and the largest `bbox.xmax` and `bbox.ymax` give you the area that the whole row group covers.

The small function below does this for one row group. Use it for **every** row group, and test each one against Snowbird:

In [18]:
def covered_area(group):
    """Work out the area that one row group covers, from the statistics of its bbox columns."""
    stats = {group.column(i).path_in_schema: group.column(i).statistics for i in range(group.num_columns)}
    return box(stats["bbox.xmin"].min, stats["bbox.ymin"].min, stats["bbox.xmax"].max, stats["bbox.ymax"].max)

n_groups = buildings.metadata.num_row_groups
extents = [covered_area(buildings.metadata.row_group(i)) for i in range(n_groups)]
row_groups = gpd.GeoDataFrame({"row_group": range(n_groups)}, geometry=extents, crs="EPSG:4326")

needed = row_groups[row_groups.intersects(snowbird)]
print(len(needed), "of", n_groups, "row groups cover Snowbird:", list(needed["row_group"]))

3 of 256 row groups cover Snowbird: [109, 111, 122]


The buildings take up many times more files and gigabytes than the land use. The layout is the same, though: you only had to change the path.

In [ ]:
m = row_groups.explore(style_kwds={"fillOpacity": 0.1})
gpd.GeoDataFrame(geometry=[snowbird], crs="EPSG:4326").explore(m=m, color="red")

### 🗺️ Step 4: Read and map the buildings

Row groups let you skip **rows**. Columns let you skip **attributes**: you ask only for the ones you need. Read the row groups you found, with three columns. A row group covers a larger area than Snowbird, so the last line keeps only the buildings inside your box:

In [ ]:
columns = ["id", "height", "geometry"]
west, south, east, north = snowbird_bbox

table = buildings.read_row_groups(list(needed["row_group"]), columns=columns)

snowbird_buildings = gpd.GeoDataFrame(
    table.drop_columns("geometry").to_pandas(),
    geometry=gpd.GeoSeries.from_wkb(table["geometry"].to_pandas(), crs="EPSG:4326"),
).cx[west:east, south:north]

print(table.num_rows, "buildings read,", len(snowbird_buildings), "of them in the Snowbird area")
snowbird_buildings.explore(tooltip=["height"])

The `geometry` column holds **WKB**, the binary format you met in Example 1. `gpd.GeoSeries.from_wkb()` decodes it into shapes. Look back at what was left at each level:

In [21]:
print(f"All buildings:      {len(footprints)} files, {total_size / 1e9:.0f} GB")
print(f"After the index:    {len(covering_files)} file(s), {file_size / 1e6:.0f} MB")
print(f"After the footer:   {len(needed)} of {n_groups} row groups")
print(f"After the columns:  {len(columns)} of {len(buildings.schema_arrow.names)} columns")
print(f"Buildings read:     {table.num_rows:,} of {buildings.metadata.num_rows:,} in the file")

All buildings:      512 files, 277 GB
After the index:    1 file(s), 666 MB
After the footer:   3 of 256 row groups
After the columns:  3 of 24 columns
Buildings read:     64,181 of 4,865,057 in the file


::: {.callout-note}
## Note

💡 Remember...

A reader can skip data at every level, and each level uses a small description, not the data itself:

- 🗂️ **Folders**: `key=value` partitions let you skip whole themes and types.
- 🔎 **Files**: an index of bounding boxes lets you skip whole files.
- 🧱 **Row groups**: the footer statistics of the `bbox` columns let you skip blocks outside your area.
- 📊 **Columns**: you read only the attributes you need.

The file and row-group levels only work because the data was **sorted** by location before writing. Here, you did all of this by hand. In the DuckDB notebook, you write a single query, and DuckDB does it for you.
:::

<hr>

<a id="example3"></a>

## Exercise 1️⃣ : Vineyards from EuroCrops

[EuroCrops](https://github.com/maja601/EuroCrops) collects the field boundaries that farmers in many EU countries declare each year, together with the crop grown on each field. The crop names are **harmonised**, so the same crop has the same name in every country. [Source Cooperative](https://source.coop/cholmes/eurocrops) publishes the dataset as GeoParquet.

Now it's your turn. Use what you learned in Examples 1 and 2 to map **vineyards in Austria** 🍇. Start by listing the files:

In [22]:
eurocrops_path = "us-west-2.opendata.source.coop/cholmes/eurocrops/unprojected/geoparquet"
eurocrops_files = s3.ls(eurocrops_path)

print([path.split("/")[-1] for path in eurocrops_files])

austria_file = eurocrops_files[0]
print(f"Austria: {austria_file.split('/')[-1]}, {s3.size(austria_file) / 1e6:.0f} MB")

['AT_2021_EC21.parquet', 'BE_VLG_2021_EC21.parquet', 'DE_LS_2021_EC21.parquet', 'DE_NRW_2021_EC21.parquet', 'DK_2019_EC21.parquet', 'EE_2021_EC21.parquet', 'ES_NA_2020_EC21.parquet', 'FR_2018_EC21.parquet', 'HR_2020_EC21.parquet', 'LT_2021_EC.parquet', 'LV_2021_EC21.parquet', 'NL_2020_EC21.parquet', 'RO_ny_EC21.parquet', 'SE_2021_EC21.parquet', 'SI_2021_EC21.parquet', 'SK_2021_EC21.parquet']
Austria: AT_2021_EC21.parquet, 1160 MB


There is one file per country or region, named after its country code. Overture splits its data by theme and type. EuroCrops splits by **country**, because most users want the fields of one country.

::: {.callout-warning icon=false}
## Instructions

- ➡️ 1. Open the Austrian file with `pq.ParquetFile()` and call it `austria`. How many fields and row groups does it hold? Is there a `bbox` column?<br>
- ➡️ 2. Read row group **2** with only two columns: `EC_hcat_n` (the crop name) and `geometry`. Call the result `table`.<br>
- ➡️ 3. Keep only the vineyards and show them on a map.

**Hints**

- Tasks 1 and 2 work as in Example 2: `pq.ParquetFile(..., filesystem=s3)`, `.metadata.num_rows`, `.metadata.num_row_groups`, `.schema_arrow.names` and `.read_row_groups([...], columns=[...])`.
- For task 3, filter the table `fields` from the checkpoint cell: `fields[fields["EC_hcat_n"] == vineyard_name]`. Then call `.explore()`.
:::

#### ➡️ 1. Open the file for Austria

In [23]:
austria = pq.ParquetFile(austria_file, filesystem=s3)
print("Fields:      ", austria.metadata.num_rows)
print("Row groups:  ", austria.metadata.num_row_groups)
print("Rows per group:  ", austria.metadata.row_group(0).num_rows)
print("Columns:     ", austria.schema_arrow.names)
print("bbox column? ", "bbox" in austria.schema_arrow.names)

Fields:       2610510
Row groups:   40
Rows per group:   65536
Columns:      ['fid', 'FS_KENNUNG', 'SNAR_BEZEI', 'SL_FLAECHE', 'GEO_ID', 'INSPIRE_ID', 'GML_ID', 'GML_IDENTI', 'SNAR_CODE', 'GEO_PART_K', 'LOG_PKEY', 'GEOM_DATE_', 'FART_ID', 'GEO_TYPE', 'GML_LENGTH', 'EC_trans_n', 'EC_hcat_n', 'EC_hcat_c', 'geometry']
bbox column?  False


The file holds about 2.6 million fields in 40 row groups. The columns that start with `EC_` were added by EuroCrops, and `EC_hcat_n` holds the harmonised crop name. Unlike Overture, the file has **no `bbox` column**.

#### ➡️ 2. Read one row group, with only two columns

In [24]:
table = austria.read_row_groups([2], columns=["EC_hcat_n", "geometry"])

print(f"{table.num_rows} fields, {table.nbytes / 1e6:.1f} MB in memory")

65536 fields, 28.0 MB in memory


EuroCrops has no `bbox` column, so the footer cannot tell you where a row group lies. Row group 2 is simply a sample that holds many vineyards.

::: {.callout-important}
## ✅ Checkpoint

Run the next cell, even if you solved tasks 1 and 2 yourself. It makes sure everyone continues with the same fields. It reads the row group, turns the WKB geometries into shapes as in Example 2, and counts the fields per crop.
:::

In [25]:
austria = pq.ParquetFile(austria_file, filesystem=s3)
table = austria.read_row_groups([2], columns=["EC_hcat_n", "geometry"])

fields = gpd.GeoDataFrame(
    table.drop_columns("geometry").to_pandas(),
    geometry=gpd.GeoSeries.from_wkb(table["geometry"].to_pandas(), crs="EPSG:4326"),
)

vineyard_name = "vineyards_wine_vine_rebland_grapes"
fields["EC_hcat_n"].value_counts().head(10)

EC_hcat_n
pasture_meadow_grassland_grass        28754
grain_maize_corn_popcorn               8864
vineyards_wine_vine_rebland_grapes     8432
fallow_land_not_crop                   3827
tree_wood_forest                       2567
green_silo_maize                       1802
pumpkin_squash_gourd                   1557
soy_soybeans                           1390
not_known_and_other                    1148
winter_barley                          1019
Name: count, dtype: int64

#### ➡️ 3. Keep only the vineyards and map them

In [ ]:
vineyards = fields[fields["EC_hcat_n"] == vineyard_name]

print(len(vineyards), "vineyards out of", len(fields), "fields")
vineyards.explore(color="purple")

::: {.callout-tip collapse="true"}
## Reflection and answers

- ➡️ 1. How much of the file did you read to make this map?<br>
    ✏️ : One of 40 row groups, and only 2 of its 19 columns: a small fraction of the file.
- ➡️ 2. Imagine you want only the vineyards around **Vienna**. Could you find the right row groups from the footer, as you did for Snowbird? What would the provider have to change?<br>
    ✏️ : No. Without a `bbox` column, the footer says nothing about location, so you would have to read all 40 row groups and test every field. Two changes would fix this: add a **`bbox` column**, and **sort** the fields by location before writing, so that each row group covers a compact area, as in Overture Maps.
:::

<hr>

<a id='example4'></a>

## Takeaways

You worked with two large vector datasets in the cloud without downloading them. Folder names, an index and file footers led you to the small part you needed, and you read only that part.

**What you learned**

* 📋 **Footer first:** a Parquet file keeps its description in a footer at the end of the file. `pq.ParquetFile()` reads only this footer, a tiny share of the file, and tells you the rows, row groups and columns.
* 🌍 **GeoParquet:** the `geo` entry in the footer names the geometry column, its encoding (WKB) and the CRS. `gpd.GeoSeries.from_wkb()` turns the stored bytes into shapes you can map.
* 📊 **Statistics:** for every column in every row group, the footer stores the smallest value, the largest value and the number of empty values. The statistics of the `bbox` columns tell you which area a row group covers.
* ✂️ **Four levels of skipping:** folders (`key=value` partitions), files (an index of bounding boxes), row groups (statistics) and columns. At each level, a small description stands in for the data.
* 🛠️ **Two ways to read:** `gpd.read_parquet(..., bbox=..., columns=[...])` does the skipping for you. With `pq.ParquetFile()` and `read_row_groups()`, you do it by hand and see each step.

**Next steps**

Here, you did the skipping by hand, one file at a time. *Discover Dask* shows you how to compute on data that doesn't fit into memory. *Discover DuckDB* returns to Overture Maps: you write one SQL query, and DuckDB reads the footers and skips folders, row groups and columns for you.

📚 *Further reading:* [Apache Parquet](https://parquet.apache.org/docs/) · [GeoParquet specification](https://geoparquet.org) · [Overture Maps docs](https://docs.overturemaps.org) · [EuroCrops](https://github.com/maja601/EuroCrops) · [Cloud-Optimized Geospatial Formats Guide](https://guide.cloudnativegeo.org)

<hr>

&copy; 2026 thriveGEO GmbH
